### Tworzenie modelu regresji liniowej

In [68]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

df = pd.read_csv('dane_gotowe_ML.csv')

df.head()

,Cena,Przebieg,Pojemność skokowa,Moc,Wiek,Rodzaj paliwa_Benzyna+LPG,Rodzaj paliwa_Diesel,Rodzaj paliwa_Elektryczny,Rodzaj paliwa_Hybryda,Rodzaj paliwa_Hybryda Plug-in,...,Marka_Peugeot,Marka_Porsche,Marka_Renault,Marka_Seat,Marka_Skoda,Marka_Subaru,Marka_Suzuki,Marka_Toyota,Marka_Volkswagen,Marka_Volvo
0,327809,1,0,292,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,59000,118760,1969,254,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,80000,120000,1998,252,7,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,239000,22,0,1020,4,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,74900,5,1499,136,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Sprawdzanie ważności cech

In [69]:
korelacje = df.corr()['Cena'].sort_values(ascending=False)
print("\n--- TOP CECHY WPŁYWAJĄCE NA CENĘ ---")
print(korelacje.head(10)) 
print("\n--- CECHY OBNIŻAJĄCE CENĘ ---")
print(korelacje.tail(5))


--- TOP CECHY WPŁYWAJĄCE NA CENĘ ---
Cena                            1
Moc                             1
Pojemność skokowa               0
Marka_Mercedes-Benz             0
Rodzaj paliwa_Hybryda Plug-in   0
Marka_Porsche                   0
Marka_BMW                       0
Marka_Inne                      0
Marka_Land                      0
Rodzaj paliwa_Elektryczny       0
Name: Cena, dtype: float64

--- CECHY OBNIŻAJĄCE CENĘ ---
Rodzaj paliwa_Benzyna+LPG   -0
Marka_Opel                  -0
Skrzynia biegów_Manualna    -1
Przebieg                    -1
Wiek                        -1
Name: Cena, dtype: float64


Podział danych na X (cechy) i Y (czyli cel - Cena)

In [70]:
X = df.drop(columns=['Cena'])
y = df['Cena']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

prognozy_cen = model.predict(X_test)

mae = mean_absolute_error(y_test, prognozy_cen)
rmse = np.sqrt(mean_squared_error(y_test, prognozy_cen)) 
r2 = r2_score(y_test, prognozy_cen)

print("WYNIKI MODELU")
print(f"Średni Błąd (MAE): {mae:,.0f} PLN (O tyle średnio model myli się na aucie)")
print(f"Błąd Ważony (RMSE): {rmse:,.0f} PLN (Kara za drastyczne pomyłki w drogich autach)")
print(f"Skuteczność (R²): {r2 * 100:.1f} % (Jaki % zmienności cen ogarnia model)")

WYNIKI MODELU
Średni Błąd (MAE): 30,888 PLN (O tyle średnio model myli się na aucie)
Błąd Ważony (RMSE): 50,005 PLN (Kara za drastyczne pomyłki w drogich autach)
Skuteczność (R²): 70.5 % (Jaki % zmienności cen ogarnia model)


In [71]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': prognozy_cen,
    'Pomyłka (Błąd)': np.abs(y_test.values - prognozy_cen)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(3))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000           112,237          54,237
1                      6500           -11,815          18,315
2                    116900           121,370           4,470


### Regresja wielomianowa

In [72]:
kolumny_numeryczne = ['Przebieg', 'Pojemność skokowa', 'Moc', 'Wiek']

num_transformer = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, kolumny_numeryczne)
    ],
    remainder='passthrough'
)

model_poly = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

print("Trenowanie modelu...")
model_poly.fit(X_train, y_train)

prognozy_poly = model_poly.predict(X_test)

mae_poly = mean_absolute_error(y_test, prognozy_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, prognozy_poly))
r2_poly = r2_score(y_test, prognozy_poly)

print(f"Średni Błąd (MAE): {mae_poly:,.0f} PLN")
print(f"Błąd Ważony (RMSE): {rmse_poly:,.0f} PLN")
print(f"Skuteczność (R²): {r2_poly * 100:.1f} %")

Trenowanie modelu...
Średni Błąd (MAE): 21,842 PLN
Błąd Ważony (RMSE): 38,138 PLN
Skuteczność (R²): 82.8 %


In [73]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': prognozy_poly,
    'Pomyłka (Błąd)': np.abs(y_test.values - prognozy_poly)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(3))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000            95,829          37,829
1                      6500             1,809           4,691
2                    116900           149,950          33,050


### Regresja Random Forest 

In [74]:
from sklearn.ensemble import RandomForestRegressor


param_distributions = {
    'n_estimators': [100, 200, 300, 400],           
    'max_depth': [None, 10, 15, 20, 30],           
    'min_samples_split': [2, 5, 10],                
    'min_samples_leaf': [1, 2, 4, 6]                
}

rf_model = RandomForestRegressor(
    random_state=42, 
    oob_score=True,
    n_jobs=-1
)

random_RF_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=15,               
    cv=3,                    
    scoring='neg_mean_absolute_error', 
    random_state=42,
    n_jobs=-1,               
    verbose=2                
)

random_RF_search.fit(X_train, y_train)

best_rf_model = random_RF_search.best_estimator_

print("\n Najlepsze parametry:")
print(random_RF_search.best_params_)

y_pred = best_rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) 
r2 = r2_score(y_test, y_pred)

print("\n WYNIKI MODELU RANDOM FOREST")
print(f"OOB Score (Walidacja w locie): {best_rf_model.oob_score_:.4f}")
print(f"Skuteczność na teście (R²):   {r2 * 100:.1f} %")
print(f"Średni Błąd (MAE):             {mae:,.0f} PLN")
print(f"Błąd Ważony (RMSE):            {rmse:,.0f} PLN")

importances = best_rf_model.feature_importances_
feature_names = X_train.columns

df_importances = pd.DataFrame({'Cecha': feature_names, 'Ważność': importances})
df_importances = df_importances.sort_values(by='Ważność', ascending=False)

print("\n TOP 5 NAJWAŻNIEJSZYCH CECH")
print(df_importances.head(5).to_string(index=False))

Fitting 3 folds for each of 15 candidates, totalling 45 fits

 Najlepsze parametry:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None}

 WYNIKI MODELU RANDOM FOREST
OOB Score (Walidacja w locie): 0.8872
Skuteczność na teście (R²):   88.6 %
Średni Błąd (MAE):             15,509 PLN
Błąd Ważony (RMSE):            31,059 PLN

 TOP 5 NAJWAŻNIEJSZYCH CECH
              Cecha  Ważność
               Wiek        0
                Moc        0
  Pojemność skokowa        0
           Przebieg        0
Marka_Mercedes-Benz        0


In [75]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': y_pred,
    'Pomyłka (Błąd)': np.abs(y_test.values - y_pred)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(5))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000            65,909           7,909
1                      6500            10,651           4,151
2                    116900           132,756          15,856
3                     22900            23,199             299
4                    219900           278,514          58,614


### Regresja Gradient Boosted Trees

In [76]:
from sklearn.ensemble import GradientBoostingRegressor

gbt_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,        
    max_depth=4,              
    min_samples_leaf=2,
    random_state=42)

gbt_model.fit(X_train, y_train)

y_pred_gbt = gbt_model.predict(X_test)

mae_gbt = mean_absolute_error(y_test, y_pred_gbt)
rmse_gbt = np.sqrt(mean_squared_error(y_test, y_pred_gbt))
r2_gbt = r2_score(y_test, y_pred_gbt)

print("\n WYNIKI MODELU GRADIENT BOOSTING")
print(f"Skuteczność na teście (R²):   {r2_gbt * 100:.1f} %")
print(f"Średni Błąd (MAE):             {mae_gbt:,.0f} PLN")
print(f"Błąd Ważony (RMSE):            {rmse_gbt:,.0f} PLN")

importances_gbt = gbt_model.feature_importances_
feature_names = X_train.columns

df_importances_gbt = pd.DataFrame({'Cecha': feature_names, 'Ważność': importances_gbt})
df_importances_gbt = df_importances_gbt.sort_values(by='Ważność', ascending=False)

print("\n TOP 5 NAJWAŻNIEJSZYCH CECH (GBT)")
print(df_importances_gbt.head(5).to_string(index=False))


 WYNIKI MODELU GRADIENT BOOSTING
Skuteczność na teście (R²):   89.5 %
Średni Błąd (MAE):             15,425 PLN
Błąd Ważony (RMSE):            29,889 PLN

 TOP 5 NAJWAŻNIEJSZYCH CECH (GBT)
               Cecha  Ważność
                Wiek        0
                 Moc        0
   Pojemność skokowa        0
            Przebieg        0
Rodzaj paliwa_Diesel        0


In [77]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': y_pred_gbt,
    'Pomyłka (Błąd)': np.abs(y_test.values - y_pred_gbt)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(5))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000            77,869          19,869
1                      6500            16,736          10,236
2                    116900           147,230          30,330
3                     22900            22,302             598
4                    219900           247,770          27,870


### Ensembling - polączenie Regresji Wielomianowej, Regresji Random Forest i Regresji Gradient Boosted Trees

In [78]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

estimators = [
    ('poly', model_poly),  
    ('rf', best_rf_model),
    ('gbt', gbt_model)    
]


stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    n_jobs=-1  
)

print("Trenowanie stacku...")
stacking_model.fit(X_train, y_train)

y_pred_stack = stacking_model.predict(X_test)

mae_stack = mean_absolute_error(y_test, y_pred_stack)
rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_stack))
r2_stack = r2_score(y_test, y_pred_stack)

print("\nWYNIKI MODELU STACKING ")
print(f"Skuteczność na teście (R²):   {r2_stack * 100:.1f} %")
print(f"Średni Błąd (MAE):             {mae_stack:,.0f} PLN")
print(f"Błąd Ważony (RMSE):            {rmse_stack:,.0f} PLN")

Trenowanie stacku...

WYNIKI MODELU STACKING 
Skuteczność na teście (R²):   89.7 %
Średni Błąd (MAE):             14,916 PLN
Błąd Ważony (RMSE):            29,573 PLN


In [79]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': y_pred_stack,
    'Pomyłka (Błąd)': np.abs(y_test.values - y_pred_stack)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(5))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000            72,549          14,549
1                      6500            12,121           5,621
2                    116900           141,006          24,106
3                     22900            21,132           1,768
4                    219900           264,307          44,407


In [80]:
import pandas as pd

auto_parametry = {
    'Przebieg': 25500,
    'Pojemność skokowa': 1000,
    'Moc': 115,
    'Wiek': 1, 
    'Rodzaj paliwa_Benzyna': 1,
    'Skrzynia biegów_Automatyczna': 1,
    'Marka_Skoda': 1 
}

nowe_auto_df = pd.DataFrame(0, index=[0], columns=X_train.columns)

for cecha, wartosc in auto_parametry.items():
    if cecha in nowe_auto_df.columns:
        nowe_auto_df.at[0, cecha] = wartosc
    elif cecha.startswith('Marka_'):
        if 'Marka_Inne' in nowe_auto_df.columns:
            nowe_auto_df.at[0, 'Marka_Inne'] = 1

wycena = stacking_model.predict(nowe_auto_df)[0]

print(f"Model auta: Skoda Fabia 1.0 Benzyna, Automat, 2024 (25.5 tys. km)")
print(f"Szacowana wartość: {wycena:,.0f} PLN")

Model auta: Skoda Fabia 1.0 Benzyna, Automat, 2024 (25.5 tys. km)
Szacowana wartość: 90,985 PLN
